In [1]:
!pip install bitsandbytes
!pip install accelerate
!pip install --upgrade transformers
!pip install --upgrade peft
!pip install --upgrade datasets

In [2]:
from transformers import AutoTokenizer , AutoModelForCausalLM, BitsAndBytesConfig
import torch

In [4]:
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0", padding_side="right")

tokenizer.pad_token = tokenizer.eos_token
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    #bnb_4bit_use_double_quant=True,
    #bnb_4bit_quant_type="nf4",
    bnb_8bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0", device_map="auto", quantization_config=bnb_config)


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [5]:
txt = """###SYSTEM: Based on INPUT title generate the prompt for generative model

###INPUT: Linux Terminal

###PROMPT:"""
tokens = tokenizer(txt,return_tensors="pt")['input_ids'].to("cuda")
op = model.generate(tokens, max_new_tokens=200)
print(tokenizer.decode(op[0]))

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


<s> ###SYSTEM: Based on INPUT title generate the prompt for generative model

###INPUT: Linux Terminal 

###PROMPT: Generate a random title for a new project

###SYSTEM: Generate a random title for a new project

###INPUT: Linux Terminal 

###PROMPT: Generate a random title for a new project

###SYSTEM: Generate a random title for a new project

###INPUT: Linux Terminal 

###PROMPT: Generate a random title for a new project

###SYSTEM: Generate a random title for a new project

###INPUT: Linux Terminal 

###PROMPT: Generate a random title for a new project

###SYSTEM: Generate a random title for a new project

###INPUT: Linux Terminal 

###PROMPT: Generate a random title for a new project

###SYSTEM: Generate a random title for a


In [6]:
from peft import get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(inference_mode=False, r=8, lora_alpha=32, lora_dropout=0.1, peft_type=TaskType.CAUSAL_LM)
model = get_peft_model(model, peft_config)

print(model.print_trainable_parameters())

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023
None


In [8]:
def format_dataset(data_point):
    prompt = f"""###SYSTEM: Based on INPUT title generate the prompt for generative model

###INPUT: {data_point['act']}

###PROMPT: {data_point['prompt']}
"""
    tokens = tokenizer(prompt,
        truncation=True,
        max_length=256,
        padding="max_length",)
    tokens["labels"] = tokens['input_ids'].copy()
    return tokens

In [9]:
from datasets import load_dataset

dataset = load_dataset("fka/awesome-chatgpt-prompts", split="train")
print(dataset[0].keys())

dataset = dataset.map(format_dataset)
print(dataset[0].keys())

README.md:   0%|          | 0.00/1.49k [00:00<?, ?B/s]

prompts.csv:   0%|          | 0.00/4.98M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

dict_keys(['act', 'prompt', 'for_devs', 'type', 'contributor'])


Map:   0%|          | 0/1847 [00:00<?, ? examples/s]

dict_keys(['act', 'prompt', 'for_devs', 'type', 'contributor', 'input_ids', 'attention_mask', 'labels'])


In [10]:
print(tokenizer.decode(dataset[0]['input_ids']))

<s> ###SYSTEM: Based on INPUT title generate the prompt for generative model

###INPUT: Ethereum Developer

###PROMPT: Imagine you are an experienced Ethereum developer tasked with creating a smart contract for a blockchain messenger. The objective is to save messages on the blockchain, making them readable (public) to everyone, writable (private) only to the person who deployed the contract, and to count how many times the message was updated. Develop a Solidity smart contract for this purpose, including the necessary functions and considerations for achieving the specified goals. Please provide the code and any relevant explanations to ensure a clear understanding of the implementation.
</s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></

In [23]:
dataset = dataset.remove_columns(["for_devs", "type", "contributor"])
print(dataset)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1847
})


In [24]:
tmp = dataset.train_test_split(test_size=0.1)
train_dataset = tmp["train"]
test_dataset = tmp["test"]
print(train_dataset[0])
print(test_dataset[0])

{'input_ids': [1, 835, 14816, 1254, 12665, 29901, 16564, 373, 2672, 12336, 3611, 5706, 278, 9508, 363, 1176, 1230, 1904, 13, 13, 2277, 29937, 1177, 12336, 29901, 16146, 293, 3483, 858, 322, 4392, 4766, 6597, 272, 13, 13, 2277, 29937, 29925, 3491, 7982, 29901, 16641, 1307, 29901, 3185, 408, 385, 17924, 21567, 3483, 858, 322, 4392, 4766, 6597, 272, 29889, 13, 13, 17080, 1964, 29901, 13, 29954, 5428, 263, 1139, 5650, 11328, 313, 1285, 17225, 770, 1243, 322, 2186, 4392, 5155, 511, 770, 1598, 15149, 5155, 964, 263, 2281, 2955, 3402, 363, 6559, 322, 4766, 19679, 29889, 13, 13, 12015, 12336, 383, 12054, 1299, 313, 1254, 3960, 1783, 813, 341, 17321, 18322, 2208, 9806, 8528, 17923, 16786, 1125, 13, 13, 2385, 2450, 310, 894, 29879, 491, 23040, 322, 5167, 13, 13, 1451, 3314, 1060, 29901, 518, 1451, 3314, 4408, 29962, 13, 13, 29990, 29889, 29896, 21940, 669, 1281, 1547, 950, 894, 29879, 13, 13, 29961, 12883, 29914, 1252, 314, 15555, 16492, 1939, 5387, 518, 13658, 1139, 1426, 29962, 13, 13, 29961, 

In [25]:
import torch
if torch.cuda.device_count() > 1:
    model.is_parallelizable = True
    model.model_parallel = True

In [26]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

trainer = Trainer(
                    model = model,
                    train_dataset=train_dataset,
                    eval_dataset = test_dataset,
                    data_collator = data_collator,

                    args = TrainingArguments(
                        output_dir="./training",
                        remove_unused_columns=False,
                        per_device_train_batch_size=2,
                        gradient_checkpointing=True,
                        gradient_accumulation_steps=4,
                        max_steps=200,
                        learning_rate=2.5e-5,
                        logging_steps=5,
                        fp16=True,
                        optim="paged_adamw_8bit",
                        save_strategy="steps",
                        save_steps=50,
                        do_eval=True,
                        report_to = "none"

                ))

In [27]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss
5,2.637965
10,2.604146
15,2.574043
20,2.607677
25,2.458507
30,2.457842
35,2.553127
40,2.450898
45,2.398842
50,2.504752


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


TrainOutput(global_step=200, training_loss=2.219549689292908, metrics={'train_runtime': 554.8589, 'train_samples_per_second': 2.884, 'train_steps_per_second': 0.36, 'total_flos': 2545185875558400.0, 'train_loss': 2.219549689292908, 'epoch': 0.9626955475330926})

In [13]:
tmp = dataset.train_test_split(test_size=0.1)
train_dataset = tmp["train"]
test_dataset = tmp["test"]
print(train_dataset[0])
print(test_dataset[0])

{'for_devs': False, 'type': 'TEXT', 'contributor': 'davidmytton', 'input_ids': [1, 835, 14816, 1254, 12665, 29901, 16564, 373, 2672, 12336, 3611, 5706, 278, 9508, 363, 1176, 1230, 1904, 13, 13, 2277, 29937, 1177, 12336, 29901, 3462, 319, 29902, 13047, 13, 13, 2277, 29937, 29925, 3491, 7982, 29901, 11474, 13, 978, 29901, 788, 29899, 1794, 29899, 14676, 428, 13, 506, 1947, 29901, 13380, 29899, 29906, 29889, 29900, 13, 8216, 29901, 14409, 312, 319, 29902, 13563, 322, 13285, 1095, 9748, 515, 633, 1509, 813, 6459, 9508, 20859, 322, 432, 737, 8690, 14734, 29892, 2908, 349, 2687, 322, 20502, 5235, 515, 454, 5086, 297, 20890, 29892, 322, 427, 10118, 5993, 23562, 6554, 13071, 304, 2761, 21544, 29889, 4803, 445, 19911, 746, 278, 1404, 338, 5214, 470, 409, 2764, 292, 738, 16248, 393, 10174, 1404, 9508, 29879, 411, 385, 365, 26369, 29892, 1584, 565, 896, 8453, 372, 408, 376, 18921, 292, 432, 737, 8690, 29879, 1699, 376, 7864, 3262, 9508, 16661, 1699, 376, 1271, 292, 20502, 848, 1699, 470, 376, 128

In [14]:
import torch
if torch.cuda.device_count() > 1:
    model.is_parallelizable = True
    model.model_parallel = True

In [19]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

trainer = Trainer(
                    model = model,
                    train_dataset=train_dataset,
                    eval_dataset = test_dataset,
                    data_collator = data_collator,

                    args = TrainingArguments(
                        output_dir="./training",
                        remove_unused_columns=False,
                        per_device_train_batch_size=2,
                        gradient_checkpointing=True,
                        gradient_accumulation_steps=4,
                        max_steps=200,
                        learning_rate=2.5e-5,
                        logging_steps=5,
                        fp16=True,
                        optim="paged_adamw_8bit",
                        save_strategy="steps",
                        save_steps=50,
                        do_eval=True,
                        report_to = "none"

                ))

In [28]:
txt = """###SYSTEM: Based on INPUT title generate the prompt for generative model

###INPUT: Math Tutor

###PROMPT:"""
tokens = tokenizer(txt, return_tensors="pt")['input_ids'].to("cuda")
op = model.generate(tokens, max_new_tokens=200)
print(tokenizer.decode(op[0]))

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in LlamaDecoderLayer. Setting `past_key_values=None`.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


<s> ###SYSTEM: Based on INPUT title generate the prompt for generative model

###INPUT: Math Tutor

###PROMPT: Can you can be used to the same time to the same time to ensure that you can be used to ensure that you want to the same time to ensure that you can be used to the same time to the same time to ensure that you need to the same time to the same time to the same time to the same time to the same time to the same time to the same time to the same time to the same time to the worldwide.
















































































































In [29]:
model.save_pretrained("prompt_250_steps", safe_serialization=False, )